In [1]:
from pyspark.sql import functions as F

silver_df = spark.table("silver.meter_readings")

print("Silver profiling note initialised.")

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 3, Finished, Available, Finished, False)

Silver profiling note initialised.


In [2]:
summary_df = silver_df.agg(
    F.count("*").alias("RowCount"),
    F.countDistinct("HouseholdID").alias("DistinctHouseholds"),
    F.min("ReadingTimestamp").alias("MinTimestamp"),
    F.max("ReadingTimestamp").alias("MaxTimestamp"),
    F.min("ConsumptionKWh").alias("MinConsumptionKWh"),
    F.max("ConsumptionKWh").alias("MaxConsumptionKWh"),
    F.avg("ConsumptionKWh").alias("AvgConsumptionKWh")
)

display(summary_df)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f0b115f9-1431-4397-b703-3f14c8f76ff8)

In [3]:
print("TARIFF DISTRIBUTION")
print("-" * 50)

silver_df.groupBy("TariffType") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 5, Finished, Available, Finished, False)

TARIFF DISTRIBUTION
--------------------------------------------------
+----------+---------+
|TariffType|count    |
+----------+---------+
|Std       |134052050|
|ToU       |33759411 |
+----------+---------+



In [4]:
household_tariff_df = (
    silver_df
    .groupBy("HouseholdID")
    .agg(
        F.countDistinct("TariffType").alias("TariffTypeCount")
    )
)

print("HOUSEHOLD TARIFF BEHAVIOUR")
print("-" * 50)

household_tariff_df.groupBy(
    "TariffTypeCount"
).count().orderBy(
    "TariffTypeCount"
).show()

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 6, Finished, Available, Finished, False)

HOUSEHOLD TARIFF BEHAVIOUR
--------------------------------------------------
+---------------+-----+
|TariffTypeCount|count|
+---------------+-----+
|              1| 5561|
+---------------+-----+



In [5]:
from pyspark.sql.window import Window

reading_window = (
    Window
    .partitionBy("HouseholdID")
    .orderBy("ReadingTimestamp")
)

interval_df = (
    silver_df
    .select(
        "HouseholdID",
        "ReadingTimestamp"
    )
    .withColumn(
        "PreviousTimestamp",
        F.lag("ReadingTimestamp").over(reading_window)
    )
    .withColumn(
        "IntervalMinutes",
        (
            F.col("ReadingTimestamp").cast("long")
            - F.col("PreviousTimestamp").cast("long")
        ) / 60
    )
    .filter(F.col("PreviousTimestamp").isNotNull())
)

print("READING INTERVAL DISTRIBUTION")
print("-" * 50)

interval_df.groupBy("IntervalMinutes") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(15, truncate=False)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 7, Finished, Available, Finished, False)

READING INTERVAL DISTRIBUTION
--------------------------------------------------
+---------------+---------+
|IntervalMinutes|count    |
+---------------+---------+
|30.0           |167780495|
|60.0           |16870    |
|1470.0         |4641     |
|2910.0         |892      |
|90.0           |724      |
|120.0          |302      |
|150.0          |248      |
|270.0          |240      |
|180.0          |215      |
|210.0          |215      |
|4350.0         |172      |
|240.0          |155      |
|300.0          |117      |
|330.0          |92       |
|5790.0         |70       |
+---------------+---------+
only showing top 15 rows



In [6]:
gap_summary = (
    interval_df
    .filter(F.col("IntervalMinutes") != 30)
    .agg(
        F.count("*").alias("Non30MinuteIntervals"),
        F.min("IntervalMinutes").alias("MinNonStandardInterval"),
        F.max("IntervalMinutes").alias("MaxNonStandardInterval"),
        F.avg("IntervalMinutes").alias("AvgNonStandardInterval")
    )
)

display(gap_summary)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 388e6345-68c0-4f70-bb3a-bf69932a0904)

In [7]:
daily_estimate_df = (
    silver_df
    .groupBy(
        "HouseholdID",
        "ReadingDate",
        "TariffType"
    )
    .agg(
        F.sum("ConsumptionKWh").alias("DailyConsumptionKWh"),
        F.count("*").alias("ReadingCount")
    )
)

daily_row_count = daily_estimate_df.count()

print("DAILY GOLD CANDIDATE")
print("-" * 50)
print(f"Rows: {daily_row_count:,}")

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 9, Finished, Available, Finished, False)

DAILY GOLD CANDIDATE
--------------------------------------------------
Rows: 3,510,403


In [8]:
print("DAILY READING COUNT DISTRIBUTION")
print("-" * 50)

daily_estimate_df.groupBy(
    "ReadingCount"
).count().orderBy(
    "ReadingCount"
).show(20, truncate=False)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 10, Finished, Available, Finished, False)

DAILY READING COUNT DISTRIBUTION
--------------------------------------------------
+------------+-----+
|ReadingCount|count|
+------------+-----+
|1           |11301|
|4           |2    |
|8           |2    |
|9           |2    |
|10          |2    |
|11          |1    |
|12          |2    |
|13          |2    |
|14          |2    |
|15          |12   |
|16          |15   |
|17          |29   |
|18          |65   |
|19          |132  |
|20          |168  |
|21          |250  |
|22          |318  |
|23          |369  |
|24          |423  |
|25          |421  |
+------------+-----+
only showing top 20 rows



In [9]:
hourly_profile_df = (
    silver_df
    .groupBy("ReadingHour")
    .agg(
        F.sum("ConsumptionKWh").alias("TotalConsumptionKWh"),
        F.avg("ConsumptionKWh").alias("AvgConsumptionKWh"),
        F.count("*").alias("ReadingCount")
    )
    .orderBy("ReadingHour")
)

display(hourly_profile_df)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d9ad3ec-c741-4d6e-bec7-424c48b34895)

In [10]:
file_profile_df = (
    silver_df
    .groupBy("SourceFileName")
    .agg(
        F.count("*").alias("RowCount"),
        F.countDistinct("HouseholdID").alias("Households"),
        F.min("ReadingTimestamp").alias("MinTimestamp"),
        F.max("ReadingTimestamp").alias("MaxTimestamp")
    )
    .orderBy("SourceFileName")
)

display(file_profile_df.limit(20))

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f2547e54-c3c2-4314-b5e2-6ee630e53351)

In [11]:
spark.sql(
    "DESCRIBE DETAIL silver.meter_readings"
).show(truncate=False)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 13, Finished, Available, Finished, False)

+------+------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+-----------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------+----------------+----------------+------------------------+
|format|id                                  |name                                                                                                                                           |description|location                                                                                                                                       |createdAt              |lastModified           |partitionColumns|clust

In [12]:
reading_count_distribution = (
    daily_estimate_df
    .groupBy("ReadingCount")
    .count()
    .orderBy(F.desc("count"))
)

print("MOST COMMON DAILY READING COUNTS")
print("-" * 50)

reading_count_distribution.show(15, truncate=False)

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 14, Finished, Available, Finished, False)

MOST COMMON DAILY READING COUNTS
--------------------------------------------------
+------------+-------+
|ReadingCount|count  |
+------------+-------+
|48          |3469352|
|47          |21209  |
|1           |11301  |
|46          |1005   |
|29          |544    |
|28          |543    |
|30          |529    |
|26          |510    |
|27          |501    |
|45          |426    |
|24          |423    |
|25          |421    |
|31          |399    |
|23          |369    |
|22          |318    |
+------------+-------+
only showing top 15 rows



In [13]:
complete_days = (
    daily_estimate_df
    .filter(F.col("ReadingCount") == 48)
    .count()
)

incomplete_days = (
    daily_estimate_df
    .filter(F.col("ReadingCount") != 48)
    .count()
)

print("DAILY COMPLETENESS")
print("-" * 50)
print(f"Complete 48-reading days: {complete_days:,}")
print(f"Other/partial days:       {incomplete_days:,}")

StatementMeta(, 17a891a3-0733-4725-9ee4-064c0d7b97fa, 15, Finished, Available, Finished, False)

DAILY COMPLETENESS
--------------------------------------------------
Complete 48-reading days: 3,469,352
Other/partial days:       41,051
